# NHẬN DẠNG TIẾNG NÓI TIẾNG VIỆT BẰNG DEEP LEARNING

**STT 19 — Automatic Speech Recognition for Vietnamese Language Using Deep Learning**  
Notebook này dùng lại toàn bộ module trong `src/`; không tạo pipeline thứ hai. Mọi output thực nghiệm bắt đầu rỗng và chỉ xuất hiện sau khi chạy cell.

## 1. Mục tiêu và phát biểu bài toán

Đầu vào là audio tiếng Việt, đầu ra là transcript. Ta so sánh Wav2Vec2-Vietnamese CTC + LM 4-gram với PhoWhisper Base fine-tuned trên VIVOS. Hai model dùng chung test rows, normalization và WER/CER.

## 2. Cơ sở lý thuyết

ASR khác TTS: ASR ánh xạ audio→text, còn TTS text→audio. Wav2Vec2 học representation trực tiếp từ waveform rồi giải mã CTC; Whisper là Transformer encoder–decoder nhận log-Mel và sinh subword token tự hồi quy. Fine-tuning cập nhật weights theo dữ liệu miền đích. Weight decay, validation và early stopping giúp hạn chế overfitting.

## 3. Công thức toán học

Attention: $\operatorname{softmax}(QK^T/\sqrt{d_k})V$.  
CTC: $P(y|x)=\sum_{\pi\in\mathcal{B}^{-1}(y)}\prod_t P(\pi_t|x)$.  
Whisper cross-entropy: $\mathcal{L}=-\sum_{t\in\Omega}\log P(y_t^*|y_{<t}^*,x)/|\Omega|$.  
$WER=(S_w+D_w+I_w)/N_w$ và $CER=(S_c+D_c+I_c)/N_c$. WER có thể lớn hơn 1 nếu insertion nhiều.

## 4. Môi trường và thiết bị

Cell dưới in thông tin thật của runtime. Trên Colab, hãy chọn GPU T4 trước khi chạy.

In [ ]:
import platform
import torch

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 5. Cài thư viện

Local: chạy `pip install -r requirements.txt` trong `.venv`. Colab thường đã có PyTorch; cài phần còn lại bằng: `pip install transformers datasets evaluate accelerate gradio soundfile scipy pyyaml`. Không cài lại hàng loạt nếu import đã thành công.

In [ ]:
from pathlib import Path
import json, sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'
config = load_config(CONFIG_PATH)
RUN_WAV2VEC2 = False
RUN_TRAINING = False
RUN_WHISPER_EVAL = False
print(json.dumps(config['data'], ensure_ascii=False, indent=2))

## 6. Tải hoặc nạp dataset

VIVOS 1.0 được tải từ https://doi.org/10.5281/zenodo.7068130 và kiểm MD5 `72972a8b14050f3f11ea7c3debabd7af`. Common Voice 27 Vietnamese tải qua Mozilla Data Collective theo điều khoản. Sau khi đặt dữ liệu, chạy `python scripts/prepare_data.py`. Không tạo data giả cho kết quả nghiên cứu.

In [ ]:
from src.data_loader import load_jsonl_manifest

processed_dir = Path(config['paths']['processed_data'])
manifest_paths = {name: processed_dir / f'{name}.jsonl' for name in ('train', 'validation', 'test')}
if all(path.exists() for path in manifest_paths.values()):
    splits = {name: load_jsonl_manifest(path) for name, path in manifest_paths.items()}
    print({name: len(rows) for name, rows in splits.items()})
else:
    splits = {}
    print('Chưa có manifest. Chạy: python scripts/prepare_data.py --config config.yaml')

## 7. Khám phá dữ liệu

Báo cáo số mẫu, tổng/mean duration, transcript rỗng và trùng. Hai histogram được sinh từ dữ liệu thật bởi script chuẩn bị.

In [ ]:
from src.data_loader import dataset_statistics

if splits:
    stats = {name: dataset_statistics(rows) for name, rows in splits.items()}
    print(json.dumps(stats, ensure_ascii=False, indent=2))
else:
    print('Bỏ qua EDA vì chưa có dữ liệu thật.')

## 8. Tiền xử lý audio và transcript

Audio được kiểm tra, chuyển mono, resample 16 kHz và từ chối nếu quá 30 giây. Transcript dùng NFC, chữ thường và gom whitespace nhưng giữ dấu tiếng Việt/dấu câu.

In [ ]:
from src.preprocessing import normalize_transcript, prepare_asr_example

print(normalize_transcript('  TÔI   yêu Tiếng Việt!  '))
if splits:
    example = prepare_asr_example(splits['train'][0], max_seconds=config['data']['max_audio_seconds'])
    print(example['audio_id'], example['audio']['array'].shape, example['duration_seconds'])

## 9. Phân chia dữ liệu và chống leakage

VIVOS test giữ nguyên official split. Validation lấy từ train theo speaker. Test không dùng chọn hyperparameter.

In [ ]:
if splits:
    train_ids = {row['audio_id'] for row in splits['train']}
    val_ids = {row['audio_id'] for row in splits['validation']}
    test_ids = {row['audio_id'] for row in splits['test']}
    assert train_ids.isdisjoint(val_ids | test_ids)
    assert val_ids.isdisjoint(test_ids)
    print('Không trùng audio_id giữa ba split.')

## 10. Wav2Vec2-Vietnamese và đánh giá

Model `nguyenvulebinh/wav2vec2-base-vietnamese-250h` dùng waveform 16 kHz và CTC beam search kết hợp LM 4-gram. Cell có cờ để tránh tự động tải/chạy model.

In [ ]:
from src.evaluate import run_model_evaluation

if RUN_WAV2VEC2 and splits:
    wav2vec2_result = run_model_evaluation(config, model_choice='wav2vec2_vietnamese', split='test')
    print(json.dumps(wav2vec2_result['metrics'], ensure_ascii=False, indent=2))
else:
    print('Đặt RUN_WAV2VEC2=True sau khi đã chuẩn bị dataset.')

## 11. Fine-tuning Whisper

Dynamic padding, padding mask -100, AdamW, mixed precision, gradient accumulation/checkpointing và validation được cấu hình trong `src/train.py`. `config.yaml` là thí nghiệm cuối trên toàn corpus; `config.quick.yaml` chỉ dùng smoke test.

In [ ]:
from src.train import run_fine_tuning

if RUN_TRAINING and splits:
    training_result = run_fine_tuning(config)
    print(json.dumps(training_result, ensure_ascii=False, indent=2))
else:
    print('Đặt RUN_TRAINING=True chỉ khi đã xác nhận GPU và cấu hình thí nghiệm.')

## 12. Đánh giá Whisper fine-tuned

Whisper fine-tuned dùng cùng test rows và normalization với Wav2Vec2. Cell sẽ thất bại rõ nếu checkpoint chưa có; không fallback ngầm sang model khác.

In [ ]:
if RUN_WHISPER_EVAL and splits:
    whisper_result = run_model_evaluation(config, model_choice='whisper_fine_tuned', split='test')
    print(json.dumps(whisper_result['metrics'], ensure_ascii=False, indent=2))
else:
    print('Đặt RUN_WHISPER_EVAL=True sau khi có best checkpoint thật.')

## 13. So sánh kết quả

Bảng chỉ đọc metric JSON thật. Nếu thiếu file, không vẽ cột giả hoặc điền 0.

In [ ]:
import pandas as pd

metrics_dir = Path(config['paths']['metrics_dir'])
rows = []
for choice in ('wav2vec2_vietnamese', 'whisper_fine_tuned'):
    path = metrics_dir / f'{choice}_metrics.json'
    if path.exists():
        values = json.loads(path.read_text(encoding='utf-8'))
        rows.append({'model': choice, 'WER': values['wer'], 'CER': values['cer'], 'samples': values['num_samples']})
comparison = pd.DataFrame(rows)
display(comparison if not comparison.empty else pd.DataFrame({'status': ['Chưa có metric thật']}))

## 14. Phân tích lỗi

CSV prediction chứa audio ID, reference, prediction, per-sample WER/CER và nhãn thiếu/thừa/thay thế. Sai dấu hoặc ảnh hưởng vùng miền chỉ kết luận sau khi nghe mẫu.

In [ ]:
prediction_path = Path(config['paths']['predictions_dir']) / 'whisper_fine_tuned_predictions.csv'
if prediction_path.exists():
    errors = pd.read_csv(prediction_path)
    display(errors.sort_values('wer', ascending=False).head(10))
else:
    print('Chưa có prediction Whisper fine-tuned thật để phân tích.')

## 15. Biểu đồ và bảng số liệu

Hiển thị histogram dataset hoặc metric comparison khi artifact tồn tại.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

figure_path = Path(config['paths']['figures_dir']) / 'audio_duration_distribution.png'
if figure_path.exists():
    display(Image(filename=str(figure_path)))
loss_figure = Path(config['paths']['figures_dir']) / 'training_validation_loss.png'
if loss_figure.exists():
    display(Image(filename=str(loss_figure)))
if not comparison.empty:
    comparison.set_index('model')[['WER', 'CER']].plot(kind='bar', ylim=(0, None), title='WER/CER thực tế')
    plt.tight_layout(); plt.show()

## 16. Lưu mô hình

Trainer lưu checkpoints định kỳ và best model tại đường dẫn cấu hình. Weights lớn bị `.gitignore`; lưu sang Drive/Hugging Face riêng theo quyền truy cập phù hợp.

In [ ]:
best_checkpoint = Path(config['paths']['best_checkpoint'])
print('Best checkpoint exists:', best_checkpoint.is_dir(), best_checkpoint)

## 17. Chạy thử inference

Sử dụng đúng lớp mà demo gọi; thay `sample_path` bằng file thật trong manifest.

In [ ]:
from src.inference import load_transcriber

if RUN_WAV2VEC2 and splits:
    transcriber = load_transcriber('wav2vec2_vietnamese', config)
    sample_output = transcriber.transcribe_file(splits['test'][0]['audio_path'])
    print(sample_output)

## 18. Chạy demo

Local: `python scripts/launch_demo.py --port 7860`. Trên Colab có thể dùng `build_demo().launch(share=True)` nhưng share tunnel đưa giao diện ra internet; chỉ dùng với audio được phép chia sẻ.

## 19. Kết luận và hạn chế

Pipeline bảo đảm chung preprocessing/metrics, official test và không bịa kết quả. Thí nghiệm cuối fine-tune Whisper trên toàn bộ train split và đánh giá cả hai model trên toàn bộ test split; các metric được đọc trực tiếp từ artifact. Hạn chế hiện tại: read speech sạch, một epoch/một seed, tối đa 30 giây, GPU 6 GB và chưa có long-form chunk overlap.

## 20. Tài liệu tham khảo

1. Radford et al., *Robust Speech Recognition via Large-Scale Weak Supervision*, 2022. https://arxiv.org/abs/2212.04356  
2. Luong & Vu, *A non-expert Kaldi recipe for Vietnamese Speech Recognition System*, 2016. https://aclanthology.org/W16-5207/  
3. Hugging Face Whisper docs. https://huggingface.co/docs/transformers/model_doc/whisper  
4. Baevski et al., *wav2vec 2.0*, 2020. https://arxiv.org/abs/2006.11477  
5. Vietnamese Wav2Vec2 model card. https://huggingface.co/nguyenvulebinh/wav2vec2-base-vietnamese-250h  
6. Mozilla Common Voice datasets. https://commonvoice.mozilla.org/vi/datasets